# NeuralOps Attention Surgery + LoRA

A minimal end-to-end DistilBERT workflow: replace attention, inject LoRA, run a KD-assisted update, and perform inference.

In [1]:
import copy
import torch
import torch.nn.functional as F
import neuralops
from transformers import DistilBertConfig, DistilBertForSequenceClassification

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = DistilBertConfig(
    vocab_size=101, dim=64, hidden_dim=128, n_layers=1, n_heads=4, num_labels=2
)
teacher = DistilBertForSequenceClassification(config).to(device).eval()
student = copy.deepcopy(teacher)
student = neuralops.replace_attention(
    student, "cosformer", copy_weights=False,
    inject_lora=True, lora_rank=4, lora_alpha=8.0, lora_dropout=0.0,
).to(device)
print("device:", device, "attention:", student.config.neuralops_attention)


/mnt/d/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-19 19:57:25,418 [INFO] Successfully injected 'cosformer' into 1 transformer blocks.


device: cuda attention: cosformer


In [2]:
attention_names = [
    name for name, module in student.named_modules()
    if "attention" in name and module.__class__.__name__ == "UniversalAttentionWrapper"
]
lora_names = [name for name, _ in student.named_parameters() if "lora_" in name.lower()]
assert attention_names and lora_names
print("replaced attention:", attention_names)
print("LoRA parameters:", len(lora_names))


replaced attention: ['distilbert.transformer.layer.0.attention']
LoRA parameters: 10


In [3]:
input_ids = torch.randint(0, config.vocab_size, (2, 8), device=device)
attention_mask = torch.ones_like(input_ids)
labels = torch.randint(0, 2, (2,), device=device)
with torch.no_grad():
    teacher_logits = teacher(input_ids=input_ids, attention_mask=attention_mask).logits
student.train()
optimizer = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=1e-3)
student_logits = student(input_ids=input_ids, attention_mask=attention_mask).logits
from neuralops.trainer.distillation.multi_level_kd import LogitDistillationLoss
kd_loss = LogitDistillationLoss(mode="forward_kl", temperature=2.0)(
    student_logits, teacher_logits.detach()
)
loss = kd_loss + F.cross_entropy(student_logits, labels)
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
grads = [p.grad for p in student.parameters() if p.requires_grad and p.grad is not None]
assert grads and all(torch.isfinite(g).all() for g in grads)
print("KD loss:", float(kd_loss.detach()), "total loss:", float(loss.detach()), "finite gradients:", True)


KD loss: 1.728534698486328e-05 total loss: 0.7042903900146484 finite gradients: True


In [4]:
student.eval()
with torch.no_grad():
    prediction = student(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=-1)
assert torch.isfinite(prediction.float()).all()
print("predictions:", prediction.tolist())


predictions: [0, 0]
